# VOICEVOX CORE on a real GPU — the reference for the CUDA shim`voicevox_emu_cpp` runs the **CUDA** build of `voicevox_onnxruntime` with no GPU atall, against stand-in `libcudart` / `cuBLAS` / `cuDNN` that do the arithmetic onthe CPU. The question this notebook answers is the only one that matters aboutthat: **does it get the same numbers as real CUDA?**So this runs the same two programs — `cudaprobe` and `cudavvm`, fetched from therepository unchanged — on a real GPU against the real libraries, and prints thenumbers and a WAV. Compare those against the local run.**Runtime → Change runtime type → GPU** before running.

In [ ]:
!nvidia-smi -L || echo 'no GPU: Runtime -> Change runtime type -> GPU'
!nvcc --version | tail -2

## 1. The piecesThe same versions the repository pins: `voicevox_onnxruntime` 1.17.3 (the CUDAbuild, 103 MB), CORE 0.16.4, ずんだもん's voice model, and the Open JTalkdictionary.

In [ ]:
import os, subprocess, textwrap
os.makedirs('/content/vv', exist_ok=True)
os.chdir('/content/vv')

VV_ORT   = '1.17.3'
VV_CORE  = '0.16.4'
VV_VVM   = '0.1.1'
VVM      = '0.vvm'

def get(url, out):
    if os.path.exists(out):
        print('have', out); return
    print('fetching', out)
    subprocess.run(['curl', '-sfL', '-o', out, url], check=True)

get(f'https://github.com/VOICEVOX/onnxruntime-builder/releases/download/voicevox_onnxruntime-{VV_ORT}/voicevox_onnxruntime-linux-x64-cuda-{VV_ORT}.tgz', 'ort.tgz')
get(f'https://github.com/VOICEVOX/voicevox_core/releases/download/{VV_CORE}/voicevox_core-linux-x64-{VV_CORE}.zip', 'core.zip')
get(f'https://github.com/VOICEVOX/voicevox_vvm/releases/download/{VV_VVM}/{VVM}', VVM)
get('https://sourceforge.net/projects/open-jtalk/files/Dictionary/open_jtalk_dic-1.11/open_jtalk_dic_utf_8-1.11.tar.gz/download', 'ojdic.tar.gz')

!tar xzf ort.tgz && unzip -oq core.zip && tar xzf ojdic.tar.gz
!cp voicevox_onnxruntime-linux-x64-cuda-{VV_ORT}/lib/*.so* .
!cp voicevox_core-linux-x64-{VV_CORE}/lib/libvoicevox_core.so . 2>/dev/null || cp $(find . -name 'libvoicevox_core.so' | head -1) .
!cp $(find . -name 'voicevox_core.h' | head -1) . 2>/dev/null || true
!ls -la *.so* | head

## 2. A plain `.onnx` to start with`predict_duration.onnx` is the small model VOICEVOX ships for testing — noencryption, four convolutions, and the four kernels the shim implements first.Two runs: the CPU provider, which is the arithmetic everyone agrees on, and theCUDA provider on the real GPU.

In [ ]:
REPO = 'https://raw.githubusercontent.com/yomei-o/voicevox_emu_cpp/main'
for f in ['src/cudaprobe.c', 'src/cudavvm.c', 'src/onnxruntime_c_api.h', 'src/voicevox_core.h']:
    subprocess.run(['curl', '-sfL', '-o', os.path.basename(f), f'{REPO}/{f}'], check=True)
subprocess.run(['curl', '-sfL', '-o', 'predict_duration.onnx',
                f'{REPO}/guest/predict_duration.onnx'], check=True)
!gcc -O2 -Wall -I. -o cudaprobe cudaprobe.c -ldl
!gcc -O2 -Wall -I. -o cudavvm cudavvm.c -L. -lvoicevox_core -Wl,-rpath,'$ORIGIN'
!ls -la cudaprobe cudavvm

In [ ]:
print('================ CPU provider (the arithmetic everyone agrees on)')
!LD_LIBRARY_PATH=. ./cudaprobe ./libvoicevox_onnxruntime.so.{VV_ORT} ./predict_duration.onnx --cpu 2>/dev/null | tail -20

In [ ]:
print('================ CUDA provider, on the real GPU')
!LD_LIBRARY_PATH=. ./cudaprobe ./libvoicevox_onnxruntime.so.{VV_ORT} ./predict_duration.onnx 2>/dev/null | tail -20

The two should agree to within float rounding. If they do, the CUDA path issound and the shim has an unambiguous target.

## 3. The whole pipeline, on the GPU`cudavvm` is the same program the repository runs against the stand-ins: loadthe runtime, build a synthesizer with `acceleration_mode = GPU`, decrypt thevoice model, and speak. Here it does it for real.

In [ ]:
print('================ the full pipeline on the GPU')
!LD_LIBRARY_PATH=. ./cudavvm ./libvoicevox_onnxruntime.so.{VV_ORT} ./open_jtalk_dic_utf_8-1.11 ./{VVM} 3 2>&1 | grep -v '^kernel ' | tail -25

## 4. The reference WAV`cudavvm` throws its audio away — it was written to enumerate kernels, not tolisten. This produces one to compare against, with the same text and style, anda checksum so two runs can be compared without ears.

In [ ]:
say_c = r'''
#include <stdio.h>
#include <stdlib.h>
#include "voicevox_core.h"
int main(int argc, char** argv) {
    const char* ort = argv[1]; const char* dict = argv[2];
    const char* vvm = argv[3]; const char* text = argv[4];
    uint32_t style = (uint32_t)atoi(argv[5]); const char* out = argv[6];
    VoicevoxLoadOnnxruntimeOptions o = voicevox_make_default_load_onnxruntime_options();
    o.filename = ort;
    const VoicevoxOnnxruntime* rt = NULL;
    if (voicevox_onnxruntime_load_once(o, &rt)) { puts("load_once failed"); return 1; }
    OpenJtalkRc* ojt = NULL;
    if (voicevox_open_jtalk_rc_new(dict, &ojt)) { puts("open_jtalk failed"); return 1; }
    VoicevoxInitializeOptions io = voicevox_make_default_initialize_options();
    io.acceleration_mode = VOICEVOX_ACCELERATION_MODE_GPU;
    VoicevoxSynthesizer* syn = NULL;
    if (voicevox_synthesizer_new(rt, ojt, io, &syn)) { puts("synthesizer_new failed"); return 1; }
    printf("gpu_mode=%d\n", (int)voicevox_synthesizer_is_gpu_mode(syn));
    VoicevoxVoiceModelFile* m = NULL;
    if (voicevox_voice_model_file_open(vvm, &m)) { puts("model open failed"); return 1; }
    if (voicevox_synthesizer_load_voice_model(syn, m)) { puts("load_voice_model failed"); return 1; }
    VoicevoxTtsOptions t = voicevox_make_default_tts_options();
    uintptr_t len = 0; uint8_t* wav = NULL;
    VoicevoxResultCode r = voicevox_synthesizer_tts(syn, text, style, t, &len, &wav);
    if (r) { printf("tts failed %d\n", (int)r); return 1; }
    FILE* f = fopen(out, "wb"); fwrite(wav, 1, len, f); fclose(f);
    printf("wrote %s %zu bytes\n", out, (size_t)len);
    return 0;
}
'''
open('gpusay.c', 'w').write(say_c)
!gcc -O2 -Wall -I. -o gpusay gpusay.c -L. -lvoicevox_core -Wl,-rpath,'$ORIGIN'
!LD_LIBRARY_PATH=. ./gpusay ./libvoicevox_onnxruntime.so.{VV_ORT} ./open_jtalk_dic_utf_8-1.11 ./{VVM} 'あ' 3 gpu_a.wav 2>&1 | tail -3
!LD_LIBRARY_PATH=. ./gpusay ./libvoicevox_onnxruntime.so.{VV_ORT} ./open_jtalk_dic_utf_8-1.11 ./{VVM} 'ずんだもんなのだ' 3 gpu_zundamon.wav 2>&1 | tail -3

In [ ]:
import hashlib, struct
for name in ['gpu_a.wav', 'gpu_zundamon.wav']:
    b = open(name, 'rb').read()
    n = (len(b) - 44) // 2
    s = struct.unpack('<%dh' % n, b[44:44 + n * 2])
    print(f'{name}: {len(b)} bytes, {n} samples, peak {max(abs(v) for v in s)}, '
          f'sha256 {hashlib.sha256(b).hexdigest()[:16]}')

In [ ]:
from IPython.display import Audio, display
for name in ['gpu_a.wav', 'gpu_zundamon.wav']:
    print(name)
    display(Audio(name))

## 5. Taking the numbers homeDownload these and compare against the local run:- `gpu_a.wav`, `gpu_zundamon.wav` — the audio a real GPU produces- the printed `predict_duration` values, which are the first thing a shim has  to match`tools/wavcmp.mjs` in the repository does the comparison:    node tools/wavcmp.mjs gpu_a.wav shim_a.wavA difference of a few least-significant bits is the same speech; anything youcould hear is percent, not thousandths.**Credit.** Audio made with these models carries `VOICEVOX:ずんだもん`.

In [ ]:
from google.colab import files
for f in ['gpu_a.wav', 'gpu_zundamon.wav']:
    files.download(f)